<a href="https://colab.research.google.com/github/gitBarrettJones/CIS115-Spring2026-Assignments/blob/main/WeekAI2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import sqlite3
from datetime import datetime, timedelta
import re

# Custom exception for quitting actions
class UserCancel(Exception):
    pass

# Connect to database
conn = sqlite3.connect("garden.db")
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS plants (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    type TEXT,
    last_watered TEXT,
    frequency INTEGER
)
""")
conn.commit()


# ---------- INPUT HANDLING ----------

def safe_input(prompt):
    value = input(prompt).strip()
    if value.lower() == "q":
        raise UserCancel
    return value


def get_text_only(prompt):
    while True:
        try:
            value = safe_input(prompt)

            if not value:
                print("❌ This field cannot be empty.")
                continue

            if not re.match("^[A-Za-z ]+$", value):
                print("❌ Only letters and spaces allowed.")
                continue

            return value

        except UserCancel:
            raise


def get_valid_date(prompt):
    while True:
        try:
            date_str = safe_input(prompt)
            datetime.strptime(date_str, "%m-%d-%Y")  # FIXED HERE
            return date_str
        except ValueError:
            print("❌ Use MM-DD-YYYY format (e.g., 04-23-2026).")
        except UserCancel:
            raise


def get_valid_int(prompt, min_value=1):
    while True:
        try:
            value = safe_input(prompt)
            value = int(value)

            if value < min_value:
                print(f"❌ Must be at least {min_value}.")
                continue

            return value

        except ValueError:
            print("❌ Enter a valid number.")
        except UserCancel:
            raise


def plant_exists(plant_id):
    cursor.execute("SELECT id FROM plants WHERE id = ?", (plant_id,))
    return cursor.fetchone() is not None


# ---------- CORE FUNCTIONS ----------

def add_plant():
    try:
        name = get_text_only("Plant name (or q to cancel): ")
        plant_type = get_text_only("Plant type (or q to cancel): ")
        last_watered = get_valid_date("Last watered MM-DD-YYYY (or q): ")
        frequency = get_valid_int("Watering frequency (days) (or q): ")

        cursor.execute("""
        INSERT INTO plants (name, type, last_watered, frequency)
        VALUES (?, ?, ?, ?)
        """, (name, plant_type, last_watered, frequency))

        conn.commit()
        print("✅ Plant added!\n")

    except UserCancel:
        print("↩️ Action cancelled.\n")


def view_plants():
    cursor.execute("SELECT * FROM plants")
    plants = cursor.fetchall()

    if not plants:
        print("🌱 No plants found.\n")
        return

    today = datetime.today()

    for plant in plants:
        plant_id, name, plant_type, last_watered, frequency = plant

        try:
            last_date = datetime.strptime(last_watered, "%m-%d-%Y")  # FIXED HERE
            next_water = last_date + timedelta(days=frequency)
            days_left = (next_water - today).days
        except Exception:
            print(f"⚠️ Data error for plant ID {plant_id}")
            continue

        status = "✅ OK" if days_left >= 0 else "💧 Needs Water!"

        print(f"""
ID: {plant_id}
Name: {name}
Type: {plant_type}
Next Watering: {next_water.date()}
Days Left: {days_left}
Status: {status}
------------------------
""")


def update_plant():
    try:
        plant_id = get_valid_int("Enter plant ID (or q to cancel): ")

        if not plant_exists(plant_id):
            print("❌ Plant not found.\n")
            return

        new_last_watered = get_valid_date("New date MM-DD-YYYY (or q): ")
        new_frequency = get_valid_int("New frequency (or q): ")

        cursor.execute("""
        UPDATE plants
        SET last_watered = ?, frequency = ?
        WHERE id = ?
        """, (new_last_watered, new_frequency, plant_id))

        conn.commit()
        print("✅ Plant updated!\n")

    except UserCancel:
        print("↩️ Action cancelled.\n")


def delete_plant():
    try:
        plant_id = get_valid_int("Enter plant ID (or q to cancel): ")

        if not plant_exists(plant_id):
            print("❌ Plant not found.\n")
            return

        confirm = safe_input("Are you sure? (y/n or q): ").lower()

        if confirm == "q":
            raise UserCancel

        if confirm != "y":
            print("Cancelled.\n")
            return

        cursor.execute("DELETE FROM plants WHERE id = ?", (plant_id,))
        conn.commit()

        print("🗑️ Plant deleted!\n")

    except UserCancel:
        print("↩️ Action cancelled.\n")


# ---------- MAIN MENU ----------

def main():
    while True:
        print("""
1. Add Plant
2. View Plants
3. Update Plant
4. Delete Plant
5. Exit
(Type 'q' anytime to cancel an action)
""")

        choice = input("Choose an option: ").strip()

        if choice == "1":
            add_plant()
        elif choice == "2":
            view_plants()
        elif choice == "3":
            update_plant()
        elif choice == "4":
            delete_plant()
        elif choice == "5":
            break
        else:
            print("❌ Invalid choice.\n")


if __name__ == "__main__":
    main()
    conn.close()


1. Add Plant
2. View Plants
3. Update Plant
4. Delete Plant
5. Exit
(Type 'q' anytime to cancel an action)

Choose an option: 1
Plant name (or q to cancel): q
↩️ Action cancelled.


1. Add Plant
2. View Plants
3. Update Plant
4. Delete Plant
5. Exit
(Type 'q' anytime to cancel an action)

Choose an option: 2
⚠️ Data error for plant ID 4

1. Add Plant
2. View Plants
3. Update Plant
4. Delete Plant
5. Exit
(Type 'q' anytime to cancel an action)

Choose an option: 4
Enter plant ID (or q to cancel): 4
Are you sure? (y/n or q): y
🗑️ Plant deleted!


1. Add Plant
2. View Plants
3. Update Plant
4. Delete Plant
5. Exit
(Type 'q' anytime to cancel an action)

Choose an option: 1
Plant name (or q to cancel): Watermelon
Plant type (or q to cancel): Fruit
Last watered MM-DD-YYYY (or q): 04-22-2026
Watering frequency (days) (or q): 4
✅ Plant added!


1. Add Plant
2. View Plants
3. Update Plant
4. Delete Plant
5. Exit
(Type 'q' anytime to cancel an action)

Choose an option: 1
Plant name (or q to c